# 01 — Thu Thập & Tổ Chức Dữ Liệu VSL-400

Notebook này thực hiện quy trình thu thập và chuẩn bị dữ liệu thô từ dataset
**Vietnamese Sign Language (VSL-400)**, bao gồm:

1. **Gộp nhiều phân đoạn (splits)** thành một dataset thống nhất
2. **Phân loại video** vào các thư mục gloss tương ứng
3. **Chia tập train/test** theo signer (unseen-signer evaluation)

> **Lưu ý:** Các đường dẫn trong notebook cần được điều chỉnh cho phù hợp
> với cấu trúc thư mục trên máy của bạn.

> Phần **trích xuất metadata JSON** (`extract_vsl_info`) được đặt ở
> notebook `02_data_cleaning_and_imputation.ipynb` vì cần chạy **sau khi đã
> tiền xử lý video** (TBL + Crop).

---
## 1. Import Thư Viện

In [ ]:
import os
import sys
import json
import re
import random
import shutil
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from collections import defaultdict

# Sửa lỗi encoding trên Windows
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

print("Đã import thành công tất cả thư viện cần thiết.")

---
## 2. Gộp Nhiều Phân Đoạn (Merge Splits)

Khi dataset được chia thành nhiều thư mục `split_1`, `split_2`, ..., ta cần
gộp chúng lại thành **một thư mục duy nhất** để dễ dàng xử lý.

Hỗ trợ 3 chế độ sao chép:
- `copy`: Sao chép file (mặc định)
- `hardlink`: Tạo hard link (tiết kiệm dung lượng, cùng ổ đĩa)
- `symlink`: Tạo symbolic link

In [ ]:
VIEWS = ("front_view", "left_view", "right_view")
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}

def find_splits(root):
    """Tìm tất cả thư mục split_* trong thư mục gốc."""
    root = Path(root)
    splits = [p for p in root.iterdir() if p.is_dir() and p.name.startswith("split_")]
    splits.sort(key=lambda p: int(re.sub(r"[^0-9]", "", p.name) or 0))
    return splits

def copy_file(src, dst, mode="copy", overwrite=False):
    """Sao chép file với chế độ chỉ định (copy/hardlink/symlink)."""
    dst = Path(dst)
    src = Path(src)
    dst.parent.mkdir(parents=True, exist_ok=True)
    
    if dst.exists() and not overwrite:
        return
    
    if mode == "copy":
        shutil.copy2(src, dst)
    elif mode == "hardlink":
        try:
            if dst.exists(): dst.unlink()
            os.link(src, dst)
        except Exception:
            shutil.copy2(src, dst)
    elif mode == "symlink":
        try:
            if dst.exists(): dst.unlink()
            dst.symlink_to(src)
        except Exception:
            shutil.copy2(src, dst)

def merge_json_lists(json_paths, stop_on_dup=False):
    """Gộp nhiều file JSON (danh sách metadata) thành một."""
    merged = []
    seen_ids = set()
    
    for jp in json_paths:
        jp = Path(jp)
        if not jp.exists():
            print(f"Bỏ qua file không tồn tại: {jp}")
            continue
        try:
            data = json.loads(jp.read_text(encoding="utf-8"))
        except Exception as e:
            print(f"Lỗi đọc JSON: {jp} — {e}")
            continue
        
        if isinstance(data, list):
            for item in data:
                if not isinstance(item, dict):
                    continue
                vid = None
                for k in ("video_id", "id", "name"):
                    if k in item:
                        vid = str(item[k])
                        break
                if vid is not None:
                    if stop_on_dup and vid in seen_ids:
                        raise RuntimeError(f"Trùng video_id {vid} trong {jp}")
                    if vid in seen_ids:
                        continue
                    seen_ids.add(vid)
                merged.append(item)
    
    return merged

def merge_dataset(root, out_dir, copy_mode="copy", overwrite=False, stop_on_dup=False):
    """
    Gộp tất cả thư mục split_* thành một dataset thống nhất.
    
    Parameters
    ----------
    root : str
        Thư mục gốc chứa các split_*.
    out_dir : str
        Thư mục đầu ra cho dataset đã gộp.
    copy_mode : str
        Chế độ sao chép: 'copy', 'hardlink', hoặc 'symlink'.
    """
    root = Path(root)
    out_dir = Path(out_dir)
    
    splits = find_splits(root)
    if not splits:
        print(f"Không tìm thấy thư mục split_* nào trong {root}")
        return
    
    print(f"Tìm thấy {len(splits)} splits: {[s.name for s in splits]}")
    out_dir.mkdir(parents=True, exist_ok=True)
    
    # Gộp JSON metadata theo từng view
    for view in VIEWS:
        json_paths = [s / f"{view}.json" for s in splits]
        merged_list = merge_json_lists(json_paths, stop_on_dup=stop_on_dup)
        out_json = out_dir / f"{view}.json"
        out_json.write_text(json.dumps(merged_list, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"   Gộp {view}: {len(merged_list)} entries → {out_json.name}")
    
    # Gộp file video theo từng view
    for view in VIEWS:
        out_view_dir = out_dir / view
        count = 0
        for s in splits:
            view_dir = s / view
            if not view_dir.exists():
                continue
            for src in view_dir.iterdir():
                if not src.is_file() or src.suffix.lower() not in VIDEO_EXTS:
                    continue
                dst = out_view_dir / src.name
                copy_file(src, dst, mode=copy_mode, overwrite=overwrite)
                count += 1
        print(f"   Gộp video {view}: {count} files")
    
    print(f"\n Merge hoàn tất! Đầu ra: {out_dir}")

### Chạy gộp splits

In [ ]:
# ============================================================
# CẤU HÌNH — Thay đổi đường dẫn phù hợp với máy của bạn
# ============================================================
SPLITS_ROOT = "."          # Thư mục chứa các split_1, split_2, ...
MERGED_OUTPUT = "merged"   # Thư mục đầu ra
COPY_MODE = "copy"         # "copy" | "hardlink" | "symlink"

# merge_dataset(SPLITS_ROOT, MERGED_OUTPUT, copy_mode=COPY_MODE)

---
## 3. Phân Loại Video Theo Gloss

Đọc file JSON metadata (từ dataset gốc hoặc file `front_view.json` đã gộp
ở bước 2) và phân loại từng video vào thư mục riêng theo tên gloss
(ví dụ: `Anh/`, `Chị/`, `Em/`, ...).

In [ ]:
def categorize_videos_by_gloss(json_path, src_video_dir, output_root):
    """
    Phân loại video vào thư mục theo gloss.
    
    Parameters
    ----------
    json_path : str
        Đường dẫn đến file JSON chứa metadata video.
    src_video_dir : str
        Thư mục nguồn chứa các file video.
    output_root : str
        Thư mục đích chứa các video được phân loại.
    """
    df = pd.read_json(json_path)
    print(f"Tổng số video: {len(df)}")
    print(f" Số lượng gloss: {df['gloss'].nunique()}")
    
    os.makedirs(output_root, exist_ok=True)
    
    # Hàm làm sạch tên thư mục (loại bỏ ký tự đặc biệt)
    def safe_dirname(name):
        """Loại bỏ các ký tự đặc biệt bị cấm trong tên thư mục Windows."""
        return re.sub(r'[\\/:*?"<>|]', '', name).strip()
    
    success_count = 0
    error_count = 0
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Phân loại video"):
        video_id = row.get("videoid", row.get("video_id", ""))
        gloss = row["gloss"]
        
        src_path = os.path.join(src_video_dir, f"{video_id}.mp4")
        if not os.path.exists(src_path):
            error_count += 1
            continue
        
        gloss_dir = os.path.join(output_root, safe_dirname(gloss))
        os.makedirs(gloss_dir, exist_ok=True)
        
        dst_path = os.path.join(gloss_dir, f"{video_id}.mp4")
        try:
            shutil.copy2(src_path, dst_path)
            success_count += 1
        except Exception as e:
            error_count += 1
            print(f"Lỗi: {e}")
    
    print(f"\n{'='*50}")
    print(f"Đã phân loại thành công: {success_count} video")
    print(f"Lỗi: {error_count} video")

### Chạy phân loại

In [ ]:
# ============================================================
# CẤU HÌNH — Thay đổi đường dẫn phù hợp với máy của bạn
# ============================================================
# JSON_METADATA_PATH = "merged/front_view.json"
# SRC_VIDEO_DIR = "merged/front_view"
# CATEGORIZED_OUTPUT = "VSL_FULL_FRONT_CATEGORIZED"

# categorize_videos_by_gloss(JSON_METADATA_PATH, SRC_VIDEO_DIR, CATEGORIZED_OUTPUT)

---
## 4. Chia Tập Train/Test Theo Signer (Unseen-Signer Split)

Chia dữ liệu theo **signer_id** để đảm bảo:
- Mỗi signer chỉ xuất hiện trong **một** tập duy nhất (train HOẶC test)
- Đánh giá mô hình trên những người ký chưa từng thấy trong quá trình huấn luyện
- Tỷ lệ: ~80% train / ~20% test

In [ ]:
def train_signer_count(n_signers):
    """Tính số lượng signer cho tập train (~80%)."""
    if n_signers <= 1:
        return n_signers
    n_train = round(n_signers * 0.8)
    if n_train >= n_signers:
        n_train = n_signers - 1
    if n_train <= 0:
        n_train = 1
    return n_train

def link_or_copy(src, dst):
    """Tạo hard link hoặc copy file."""
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        dst.unlink()
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)

def split_by_signer(json_path, src_videos_dir, output_root, seed=42):
    """
    Chia dữ liệu train/test theo signer ID.
    
    Parameters
    ----------
    json_path : str
        Đường dẫn file JSON chứa metadata (phải có trường 'gloss' và 'signer_id').
    src_videos_dir : str
        Thư mục chứa các file video nguồn.
    output_root : str
        Thư mục đầu ra chứa train/ và test/.
    seed : int
        Seed cho random (đảm bảo reproducibility).
    """
    json_path = Path(json_path)
    src_videos_dir = Path(src_videos_dir)
    output_root = Path(output_root)
    
    rows = json.loads(json_path.read_text(encoding="utf-8"))
    
    # Nhóm video theo gloss
    gloss_rows = defaultdict(list)
    all_signers = set()
    for r in rows:
        gloss_rows[r["gloss"]].append(r)
        all_signers.add(str(r["signer_id"]))
    
    # Shuffle signers
    random.seed(seed)
    signers_shuffled = sorted(all_signers)
    random.shuffle(signers_shuffled)
    
    # Chia signers thành train/test
    n_train_g = train_signer_count(len(signers_shuffled))
    train_signers = set(signers_shuffled[:n_train_g])
    test_signers = set(signers_shuffled[n_train_g:])
    
    assert train_signers.isdisjoint(test_signers), "Có signer trùng giữa train và test!"
    
    print(f"👥 Tổng số signer: {len(all_signers)}")
    print(f"    Train: {len(train_signers)} signers")
    print(f"    Test:  {len(test_signers)} signers")
    
    # Tạo thư mục đầu ra
    if output_root.exists():
        shutil.rmtree(output_root)
    output_root.mkdir(parents=True, exist_ok=True)
    
    # Lưu bảng phân bổ signer
    global_split_path = output_root / "global_signer_split.tsv"
    global_lines = ["signer_id\tsplit"]
    for s in sorted(train_signers):
        global_lines.append(f"{s}\ttrain")
    for s in sorted(test_signers):
        global_lines.append(f"{s}\ttest")
    global_split_path.write_text("\n".join(global_lines) + "\n", encoding="utf-8")
    
    # Phân bổ video
    train_total = test_total = 0
    
    for gloss in tqdm(sorted(gloss_rows.keys()), desc="Phân bổ video"):
        items = gloss_rows[gloss]
        
        for r in items:
            vid = str(r["video_id"])
            sid = str(r["signer_id"])
            src = src_videos_dir / f"{vid}.mp4"
            
            if not src.is_file():
                continue
            
            if sid in train_signers:
                dst_dir = output_root / "train" / gloss
                link_or_copy(src, dst_dir / f"{vid}.mp4")
                train_total += 1
            elif sid in test_signers:
                dst_dir = output_root / "test" / gloss
                link_or_copy(src, dst_dir / f"{vid}.mp4")
                test_total += 1
    
    print(f"\n{'='*50}")
    print(f"Hoàn tất chia dữ liệu!")
    print(f"    Train: {train_total} videos")
    print(f"    Test:  {test_total} videos")
    print(f"    Signer split: {global_split_path}")

### Chạy chia tập train/test

In [ ]:
# ============================================================
# CẤU HÌNH — Thay đổi đường dẫn phù hợp với máy của bạn
# ============================================================
# JSON_PATH = "merged/front_view.json"
# SRC_VIDEOS = "merged/front_view"
# SPLIT_OUTPUT = "data_splited"

# split_by_signer(JSON_PATH, SRC_VIDEOS, SPLIT_OUTPUT, seed=42)

---
## 5. Tổng Kết

Sau khi chạy xong notebook này, bạn sẽ có:

| Đầu ra | Mô tả |
|--------|-------|
| `merged/` | Dataset đã gộp từ nhiều splits |
| `VSL_FULL_FRONT_CATEGORIZED/` | Video được phân loại theo thư mục gloss |
| `data_splited/train/` | Dữ liệu huấn luyện (chia theo signer) |
| `data_splited/test/` | Dữ liệu kiểm thử (chia theo signer) |
| `data_splited/global_signer_split.tsv` | Bảng phân bổ signer ↔ train/test |

➡️ **Bước tiếp theo:** Chạy notebook `02_data_cleaning_and_imputation.ipynb`
để tiền xử lý video (TBL + Crop + Resize) và trích xuất metadata JSON.